In [221]:
import pandas as pd
import numpy as np
import os
import re

#### MovieLense Dataset Cleaning

In [222]:
PATH_MOVIELENS = os.path.join('..','dataset','MovieLens')

In [223]:
movielens = pd.read_csv(os.path.join(PATH_MOVIELENS,'movies.csv'))
movielens.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [224]:
movielens['title_clean'] =  movielens['title'].apply(lambda x : re.split(pattern = r' \(\d{4}\)',string=x)[0].lower().strip())

In [225]:
print(f'shape of the movieLens dataset is : {movielens.shape}')
print(f'moviesLens have each record unique movies? ans : {movielens['title_clean'].is_unique}')


shape of the movieLens dataset is : (62423, 4)
moviesLens have each record unique movies? ans : False


In [226]:
# Duplicate movie problem in MovieLens dataset.
# problem 1: some movies are containes duplicates record with slidly different genres pattern
# problem 2:  some containes same movies title but different released year. ex sohara (1992), sohara (2001)

# Solution : we does not know that which of this movies belong to TBMDB dataset movies, 
#            so the best approch is that to removes such suspicious movies in dataset.

# Identify duplicate movie title
duplicate_movies_title =  movielens[movielens.duplicated(subset=['title_clean'])]
duplicate_movies_title = duplicate_movies_title['title_clean'].unique().tolist()

In [227]:
# deselect those movies
movielens = movielens[~ movielens['title_clean'].isin(duplicate_movies_title)]


print(f'Verify that each movie title is unique know -> {movielens['title_clean'].is_unique}')

Verify that each movie title is unique know -> True


#### TMDB dataset cleaning

In [228]:
tmdb = pd.read_csv('../dataset/processed/movies_df_clean.csv')
tmdb.head(2)

,movie_id,budget,homepage,original_title,popularity,production_companies,production_countries,release_date,revenue,runtime,...,tagline,title,vote_average,vote_count,overview_clean,genres_clean,cast_clean,crew_clean,keywords_clean,original_language_full
0,19995,237000000,http://www.avatarmovie.com/,Avatar,150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,...,Enter the World of Pandora.,Avatar,7.2,11800,"In the 22nd century, a paraplegic Marine is di...","Fantasy, Science Fiction, Action, Adventure","Sam Worthington, Zoe Saldana, Sigourney Weave...",Director: James Cameron,"space war, future, love affair, culture clash...",English
1,285,300000000,http://disney.go.com/disneypictures/pirates/,Pirates of the Caribbean: At World's End,139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,...,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,"Captain Barbossa, long believed to be dead, ha...","Fantasy, Action, Adventure","Johnny Depp, Orlando Bloom, Keira Knightley, ...",Director: Gore Verbinski,"ocean, alliance, afterlife, traitor, east ind...",English


In [229]:
tmdb['title_clean'] = tmdb['title'].str.strip().str.lower()

In [230]:
# check duplicates in tmdb dataset
tmdb[tmdb[['title_clean']].duplicated()]

,movie_id,budget,homepage,original_title,popularity,production_companies,production_countries,release_date,revenue,runtime,...,title,vote_average,vote_count,overview_clean,genres_clean,cast_clean,crew_clean,keywords_clean,original_language_full,title_clean
2877,1255,11000000,http://www.hostmovie.com/,괴물,27.655270,"[{""name"": ""Cineclick Asia"", ""id"": 685}, {""name...","[{""iso_3166_1"": ""KR"", ""name"": ""South Korea""}]",2006-07-27,88489643,119.0,...,The Host,6.7,537,Gang-du is a dim-witted man working at his fat...,"Drama, Science Fiction, Horror","Song Kang-ho, Park Hae-il, Bae Doona",Director: Bong Joon-ho,"family, archer, river, daughter, formaldehyde...",Korean,the host
3692,10844,0,NaN,Out of the Blue,0.706355,[],"[{""iso_3166_1"": ""NZ"", ""name"": ""New Zealand""}]",2006-10-12,0,103.0,...,Out of the Blue,5.9,18,Ordinary people find extraordinary courage in ...,Drama,"Karl Urban, Tandi Wright, Simon Ferry, Matthe...",Director: Robert Sarkies,"neighbor, independent film, new zealand, gun ...",English,out of the blue
4264,2661,1377800,NaN,Batman,9.815394,"[{""name"": ""Twentieth Century Fox Film Corporat...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1966-07-30,0,105.0,...,Batman,6.1,203,The Dynamic Duo faces four super-villains who ...,"Family, Science Fiction, Crime, Comedy, Adven...","Adam West, Burt Ward, Lee Meriwether, Cesar R...",Director: Leslie H. Martinson,"shark, missile, shark attack, shark repelent,...",English,batman


In [231]:
# only 3 records are duplicate
# it safe to delete duplicate records

duplicate_movies_title =  tmdb[tmdb.duplicated(subset=['title_clean'])]
duplicate_movies_title = duplicate_movies_title['title_clean'].unique().tolist()

In [232]:
# deselect those movies
tmdb = tmdb[~ tmdb['title_clean'].isin(duplicate_movies_title)]


print(f'Verify that each movie title is unique know -> {tmdb['title_clean'].is_unique}')

Verify that each movie title is unique know -> True


In [233]:
(tmdb[['title_clean','movie_id']]
    .merge(movielens[['movieId','title_clean']],on='title_clean',how='inner')
    .rename(columns = {'movie_id':'tmdb_movieId','movieId':'movieLens_movieId'})
)

,title_clean,tmdb_movieId,movieLens_movieId
0,avatar,19995,72998
1,pirates of the caribbean: at world's end,285,53125
2,john carter,49529,93363
3,spider-man 3,559,52722
4,avengers: age of ultron,99861,122892
...,...,...,...
3032,primer,14337,8914
3033,newlyweds,72766,91829
3034,"signed, sealed, delivered",231617,164791
3035,shanghai calling,126186,101294


In [234]:
movieId_lookup =  (tmdb[['title_clean','movie_id']]
                    .merge(movielens[['movieId','title_clean']],on='title_clean',how='inner')
                    .rename(columns = {'movie_id':'tmdb_movieId','movieId':'movieLens_movieId'}))

In [235]:
movieId_lookup['title_clean'] = matched_movies['title_clean'].str.capitalize()

In [236]:
movieId_lookup

,title_clean,tmdb_movieId,movieLens_movieId
0,Avatar,19995,72998
1,Pirates of the caribbean: at world's end,285,53125
2,John carter,49529,93363
3,Spider-man 3,559,52722
4,Avengers: age of ultron,99861,122892
...,...,...,...
3032,Primer,14337,8914
3033,Newlyweds,72766,91829
3034,"Signed, sealed, delivered",231617,164791
3035,Shanghai calling,126186,101294


In [237]:
print(f'is movieLens "movieId" unique -> {movieId_lookup['movieLens_movieId'].is_unique}')
print(f'is tmdb "movieId" unique      -> {movieId_lookup['tmdb_movieId'].is_unique}')
print(f'is movie "title" unique       -> {movieId_lookup['title_clean'].is_unique}')

is movieLens "movieId" unique -> True
is tmdb "movieId" unique      -> True
is movie "title" unique       -> True


In [241]:
movieId_lookup[movieId_lookup['tmdb_movieId']==]

,title_clean,tmdb_movieId,movieLens_movieId
1206,Bridget jones's diary,634,4246


In [220]:
movieId_lookup.to_csv('../dataset/processed/movieId_lookup.csv',index=False)

In [242]:
movieId_lookup

,title_clean,tmdb_movieId,movieLens_movieId
0,Avatar,19995,72998
1,Pirates of the caribbean: at world's end,285,53125
2,John carter,49529,93363
3,Spider-man 3,559,52722
4,Avengers: age of ultron,99861,122892
...,...,...,...
3032,Primer,14337,8914
3033,Newlyweds,72766,91829
3034,"Signed, sealed, delivered",231617,164791
3035,Shanghai calling,126186,101294
